In [1]:
import numpy
import sys
import os
import matplotlib.pyplot as plt
# import skimage
import nrrd
from scipy import ndimage
import meshio
from skimage.morphology import skeletonize
from scipy.interpolate import splprep, splev
from sklearn.decomposition import PCA
from scipy.spatial import cKDTree
from scipy.interpolate import UnivariateSpline
import trimesh
from scipy.ndimage import median_filter

from scipy.spatial import cKDTree
from scipy.interpolate import UnivariateSpline
from skimage import measure

from scipy.ndimage import uniform_filter1d

sys.path.insert(0, "../src")  # add Folder_2 path to search list

from meshing_functions import getSurfaceMesh, tetra_mesh_from_stl, tetra_shell_from_two_surfaces, set_the_offset, export_centerline


pixel_spacing = numpy.loadtxt("/Users/skardova/Documents/MRI_data/Brain/Comprehensive ultrahigh resolution/ses-4712/pixel_spacing.txt")

base = "/Users/skardova/Documents/MRI_data/Brain/Comprehensive ultrahigh resolution/ses-4712/"

# offset = numpy.loadtxt("/Users/skardova/Documents/MRI_data/Brain/Comprehensive ultrahigh resolution/ses-4712/meshes/offset.txt")


in_folder = "segment_final/"    

folder = "Radial-profiles-from-mask/"

if not os.path.isdir(base + folder):
    os.mkdir(base + folder)


## Load segments, pixel spacing etc

In [2]:

filename = "Segmentation_1.seg.nrrd"
export_name = base + folder + filename[:-8]

eyeball_id = 1
muslce_ids = [2,3,4,5]
pairs = [(0,2), (1,3)]

lense_id = 7

background_id = -1


data, header = nrrd.read(base + in_folder + filename)
print(data.shape)



id_max = numpy.max(data)

print("pixel spacing = ",pixel_spacing)
print("id = ", id_max )

print("dimensions = ", data.shape)

if pixel_spacing[2]!= pixel_spacing[1]:
    print("resampling")
    factor = pixel_spacing[2]/pixel_spacing[1]
    data = ndimage.zoom(data, [1,1,factor], order = 0)


print("dimensions = ", data.shape)

id = int(numpy.max(data))
print("id = ", id)




(110, 150, 155)
pixel spacing =  [0.5 0.5 0.5]
id =  7
dimensions =  (110, 150, 155)
dimensions =  (110, 150, 155)
id =  7


## Create initial meshes

In [3]:
muscles = []

for id in range(1,numpy.max(data)+1):
    if id != background_id:
    # for id in range(3,4):
        print("id = ", id)


        if len(numpy.where((data>id-0.5) & (data<id+0.5 ))[0])>0:

            mask = numpy.zeros(data.shape, dtype=numpy.uint8)
            mask[(data>id-0.5) & (data<id+0.5 )]=1

            print(len(numpy.where(mask==1)[0]))


            numpy.save(export_name +"_filtered_surf_id_"+str(id), mask)

            if id ==eyeball_id:

                mask = ndimage.median_filter(mask, size=1)
                mask[mask<0.5]=0
                mask[mask>=0.5]=1

                mesh_surf = getSurfaceMesh(mask, export_name +"_surf_id_"+str(id)+".stl", pixel_spacing[0], True )

                mask_voxels = numpy.argwhere(mask == 1)  # shape (N, 3), coordinates in (z, y, x)
                # compute mean voxel position
                offset = mask_voxels.mean(axis=0)*pixel_spacing[0]  # [z_mean, y_mean, x_mean]
                numpy.savetxt(base + folder + "offset.txt", offset)

                mask_eye = mask

            if id in muslce_ids:
                print("Muscle")

                muscles.append(mask)
                mesh_surf = getSurfaceMesh(mask, export_name +"_surf_id_"+str(id)+".stl", pixel_spacing[0], False )

            if id == lense_id:
                mask = ndimage.median_filter(mask, size=1)
                mask[mask<0.5]=0
                mask[mask>=0.5]=1

                mesh_surf = getSurfaceMesh(mask, export_name +"_surf_id_"+str(id)+".stl", pixel_spacing[0], False )


id =  1
65767
Problem edges: 0
Outer and inner surfaces written.
id =  2
8436
Muscle
Problem edges: 4
written file:  /Users/skardova/Documents/MRI_data/Brain/Comprehensive ultrahigh resolution/ses-4712/Radial-profiles-from-mask/Segmentation_1._surf_id_2.stl
id =  3
4587
Muscle
Problem edges: 0
written file:  /Users/skardova/Documents/MRI_data/Brain/Comprehensive ultrahigh resolution/ses-4712/Radial-profiles-from-mask/Segmentation_1._surf_id_3.stl
id =  4
7222
Muscle
Problem edges: 1
written file:  /Users/skardova/Documents/MRI_data/Brain/Comprehensive ultrahigh resolution/ses-4712/Radial-profiles-from-mask/Segmentation_1._surf_id_4.stl
id =  5
6801
Muscle
Problem edges: 6
written file:  /Users/skardova/Documents/MRI_data/Brain/Comprehensive ultrahigh resolution/ses-4712/Radial-profiles-from-mask/Segmentation_1._surf_id_5.stl
id =  6
6465
id =  7
1372
Problem edges: 4
written file:  /Users/skardova/Documents/MRI_data/Brain/Comprehensive ultrahigh resolution/ses-4712/Radial-profiles-from

## Lense axis

In [4]:
mesh = trimesh.load(export_name +"_surf_id_"+str(lense_id)+".stl")
# mesh.vertices = (mesh.vertices + offset) / pixel_spacing

lense_center = mesh.vertices.mean(axis=0)

print("lense_center = ", lense_center)

X = mesh.vertices - lense_center

_, _, vh = numpy.linalg.svd(X)

axis_direction = vh[-1]  # first principal axis


def axis(t):
    return lense_center + t * axis_direction


axis_points = [axis(-20.0), axis(20.0)]

export_centerline(axis_points, 1.0 , base + folder + "lense_axis.vtp", [0,0,0])

# # # # # # # # # # # # # # # # # # # # # # # # # # # # # 

target_point =  axis(-2.25)/pixel_spacing[0]

print("target_point = ", target_point)

lense_center =  [34.74798285 63.83954948 24.45071852]
target_point =  [ 69.28843289 123.27184881  49.78635298]


## Section curve

- eyeball mesh ∩ place perpendicular to lense axis
- output: section curve

In [5]:
eyeball_mesh = trimesh.load(export_name +"_surf_id_"+str(eyeball_id)+"_outer.stl")

section = eyeball_mesh.section(
    plane_origin=target_point*pixel_spacing[0],
    plane_normal=axis_direction
)

section_curve = []

for entity in section.entities:
    idx = entity.points  # ordered indices
    
    for i in range(len(idx) - 1):
        section_curve.append([idx[i], idx[i+1]])

meshio.write(
    base + folder + "section_curve_A.vtk",
    meshio.Mesh(
        points=section.vertices,
        cells=[("line", numpy.array(section_curve))]
    )
)

## Section curve B & radial expansion


In [6]:
target_point_B =  axis(-9)/pixel_spacing[0]

section_B = eyeball_mesh.section(
    plane_origin=target_point_B*pixel_spacing[0],
    plane_normal=axis_direction
)

section_curve_B = []

for entity in section_B.entities:
    idx = entity.points  # ordered indices
    
    for i in range(len(idx) - 1):
        section_curve_B.append([idx[i], idx[i+1]])


# Center point in physical space 
center_physical = target_point_B * pixel_spacing[0]

original_vertices = numpy.array(section_B.vertices)

# Radial vectors from the center 
radial_vectors = original_vertices - center_physical

magnitudes = numpy.linalg.norm(radial_vectors, axis=1, keepdims=True)
normalized_vectors = radial_vectors / (magnitudes + 1e-8)



# Expand outward by xx mm 
expansion_distance = 1.5 
expanded_vertices = original_vertices + (normalized_vectors * expansion_distance)

section_B.vertices = expanded_vertices

# 6. Rebuild the section_curve line segments using the new expanded coordinates
expanded_section_curve = []
for entity in section_B.entities:
    idx = entity.points  # ordered indices within this specific entity
    
    for i in range(len(idx) - 1):
        # We map the local entity indices directly to our expanded_vertices array
        expanded_section_curve.append([idx[i], idx[i+1]])

meshio.write(
    base + folder + "section_curve_B.vtk",
    meshio.Mesh(
        points=section_B.vertices,
        cells=[("line", numpy.array(expanded_section_curve))]
    )
)

## Functions: 
### Fit centerlines with/without additional constraints

In [7]:
def fit_target_constrained_spline(binned_centers, target_point, order, smoothing_factor=3.5):
    """
    binned_centers: (N, 3) array of centers from the mask
    target_point: (3,) array [z, y, x] where the object should end
    """
    # 1. Append the target point to the end of your centers
    # We ensure target_point is shaped correctly for concatenation
    target_point = numpy.array(target_point).reshape(1, 3)
    constrained_points = numpy.concatenate([binned_centers, target_point], axis=0)
    
    # 2. Assign Weights (w)
    # We give the target point a very high weight (e.g., 10) 
    # and the noisy data points a lower weight (e.g., 1)
    weights = numpy.ones(len(constrained_points))
    weights[-1] = 20.0  # Force the spline to prioritize hitting the target
    
    # 3. Fit the Spline
    # We use the transpose .T because splprep expects (3, N)
    try:
        # Note: 'w' is the weights, 's' is the overall smoothing
        tck, u = splprep(constrained_points.T, w=weights, s=smoothing_factor, k=order)
        
        # 4. Evaluate from 0 to 1
        # Since the target is now the last point in the data, 
        # u=1.0 will correspond to the target point.
        u_new = numpy.linspace(-0.05, 1.0, 160)
        smooth_centerline = numpy.array(splev(u_new, tck)).T
        
        return smooth_centerline
    except Exception as e:
        print(f"Constrained spline fit failed: {e}")
        return None
    
def fit_3_constrained_spline(binned_centers, target_1, target_2, target_3, order, smoothing_factor=3.5):
    """
    Fits a spline through binned centers and 3 target guiding points,
    """
    # 1. Combine all points into one unordered pool
    target_1 = numpy.array(target_1).reshape(1, 3)
    target_2 = numpy.array(target_2).reshape(1, 3)
    target_3 = numpy.array(target_3).reshape(1, 3)
    
    unordered_points = numpy.concatenate([binned_centers, target_1], axis=0)
    
    # 2. Use PCA  to find the true main orientation axis
    pca = PCA(n_components=1)
    projection = pca.fit_transform(unordered_points)
    
    # 3. Sort ALL points based on their position along this principal axis
    sort_idx = numpy.argsort(projection.flatten())
    constrained_points = unordered_points[sort_idx]

    constrained_points = numpy.concatenate([constrained_points, target_2, target_3], axis=0)
    
    # 4. Flip check (Ensure Target 3 is at the end)
    distances_to_end = numpy.linalg.norm(constrained_points - target_3, axis=1)
    if numpy.argmin(distances_to_end) == 0:
        constrained_points = numpy.flip(constrained_points, axis=0)

    # 5. Assign Weights dynamically by finding where the targets landed
    weights = numpy.ones(len(constrained_points))
    
    for target in [target_1]:
        distances = numpy.linalg.norm(constrained_points - target, axis=1)
        matched_idx = numpy.argmin(distances)
        weights[matched_idx] = 50
    
    weights[0] = 20
    weights[-1] = 20

    # 6. Fit the Spline
    try:
        tck, u = splprep(constrained_points.T, w=weights, s=smoothing_factor, k=order)
        
        # Evaluate slightly past the boundaries to allow mathematical extension
        u_new = numpy.linspace(-0.05, 1.0, 160)
        smooth_centerline = numpy.array(splev(u_new, tck)).T
        
        return smooth_centerline
    
    except Exception as e:
        print(f"PCA-Ordered 3-point constrained spline fit failed: {e}")
        return None
    

def fit_dual_constrained_spline(binned_centers, target_1, target_2, order, smoothing_factor=3.5):
    """
    binned_centers: (N, 3) array of centers from the mask
    target_1: (3,) array where the object should begin
    target_2: (3,) array where the object should end
    """
    # 1. Prepare the sequence: [Start, Binned Data, End]
    points = [
        binned_centers,
        numpy.array(target_1).reshape(1, 3),
        numpy.array(target_2).reshape(1, 3)
    ]
    constrained_points = numpy.concatenate(points, axis=0)
    
    # 2. Assign Weights
    # Weights[0] is Start, Weights[-1] is End. 
    # High values force the spline to hit those exact coordinates.
    weights = numpy.ones(len(constrained_points))
    weights[0] = 50.0  # Force start
    weights[-1] = 50.0 # Force end
    
    # 3. Fit the Spline
    # We pass the weights array to 'w'
    try:
        tck, u = splprep(constrained_points.T, w=weights, s=smoothing_factor, k=order)
        
        # 4. Evaluate
        # u=0.0 will be target_start, u=1.0 will be target_end
        u_new = numpy.linspace(-0.05, 1.0, 160)
        smooth_centerline = numpy.array(splev(u_new, tck)).T
        
        return smooth_centerline
    except Exception as e:
        print(f"Dual constrained spline fit failed: {e}")
        return None

def fit_analytical_centerline(mask, target_point, ignored_nodes, order, n_bins=20, extension_factor=0.2):
        # 1. Get all voxel coordinates (Z, Y, X)
        points = numpy.argwhere(mask == 1).astype(float)
        
        if len(points) < 10:
            return None

        # 2. Find the main orientation using PCA
        pca = PCA(n_components=1)
        # Project points onto the first principal component 
        projection = pca.fit_transform(points)
        
        # 3. Sort points based on their position along the principal axis
        sort_idx = numpy.argsort(projection.flatten())

        sort_idx = sort_idx[int(len(sort_idx)*ignored_nodes):int(len(sort_idx)*(1-ignored_nodes))]

        sorted_points = points[sort_idx]
        sorted_projection = projection[sort_idx].flatten()

        # 4. Create "Binned Centers" to handle the noise
        # Fit to the average center of 'n' slices
        bins = numpy.linspace(sorted_projection.min(), sorted_projection.max(), n_bins)
        binned_centers = []
        
        for i in range(len(bins)-1):
            # Find points that fall within this slice of the object
            mask_slice = (sorted_projection >= bins[i]) & (sorted_projection < bins[i+1])
            if numpy.any(mask_slice):
                center = sorted_points[mask_slice].mean(axis=0)
                binned_centers.append(center)
        
        binned_centers = numpy.array(binned_centers)

        # # 5. Fit the Spline to the centers
        # # Smoothing 's' can be lower here because the binning already cleaned the noise
        tck, u = splprep(binned_centers.T, s=3.5, k=2)
        
        # # 6. Extend the object
        # # Evaluating from 0.0 to 1+extension extends the centerline
        smooth_centerline = numpy.array(splev(numpy.linspace(-extension_factor, 1.0 , 100), tck)).T

        if len(target_point)==0:
            return smooth_centerline
        
        if len(target_point)==1:
            smooth_centerline_extended = fit_target_constrained_spline(binned_centers, target_point[0], order)
            return smooth_centerline, smooth_centerline_extended
        
        if len(target_point)==2:
            smooth_centerline_extended = fit_dual_constrained_spline(binned_centers, target_point[0], target_point[1], order )
            return smooth_centerline, smooth_centerline_extended

        if len(target_point)==3:
            smooth_centerline_extended = fit_3_constrained_spline(binned_centers, target_point[0], target_point[1], target_point[2], order )
            return smooth_centerline, smooth_centerline_extended    


## Functions:
### Get insertion points ~ section curve / fitted planes

In [8]:
def fit_plane_to_two_centerlines(centerline1, centerline2):
    # 1. Combine all points from both centerlines into one cloud
    points = numpy.vstack((centerline1, centerline2))
    
    # 2. Calculate the centroid (the average point)
    centroid = numpy.mean(points, axis=0)
    
    # 3. Center the points by subtracting the centroid
    centered_points = points - centroid
    
    # 4. Use Singular Value Decomposition
    _, _, vh = numpy.linalg.svd(centered_points)
    
    # 5. The normal vector 'n' is the last row of Vh 
    # (the direction with the least variance/spread)
    normal = vh[-1]
    
    return centroid, normal/numpy.linalg.norm(normal)

def find_plane_curve_intersections(section, section_curve, plane_centroid, plane_normal):
    intersections = []
    
    # Normalize plane normal just in case
    plane_normal = plane_normal / numpy.linalg.norm(plane_normal)
    
    vertices = section.vertices
    
    for start_idx, end_idx in section_curve:
        p1 = vertices[start_idx]
        p2 = vertices[end_idx]
        
        # Signed distances to the plane
        d1 = numpy.dot(p1 - plane_centroid, plane_normal)
        d2 = numpy.dot(p2 - plane_centroid, plane_normal)
        
        # Check if the segment crosses the plane
        if d1 * d2 < 0:
            # Linear interpolation to find the exact crossing point
            t = abs(d1) / (abs(d1) + abs(d2))
            intersect_pt = p1 + t * (p2 - p1)
            intersections.append(intersect_pt)
            
    return numpy.array(intersections)



In [9]:
extended_centerlines = []
centerlines = []
extended_centerlines_1 = []

order = 3
# target_point = point on the lense axis

for muscle_id in range(len(muslce_ids)):
    mask = muscles[muscle_id]
    ignored_nodes = 0.01

    centerline, centerline_ext_1 = fit_analytical_centerline(mask, [target_point], ignored_nodes, order, n_bins=10, extension_factor=0.15)
    
    centerlines.append(centerline)
    extended_centerlines.append(centerline_ext_1)


    export_centerline(centerline_ext_1, pixel_spacing[0], base + folder + "centerline_exttended_1_id_"+str(muslce_ids[muscle_id])+".vtp", [0,0,0])
    export_centerline(centerline, pixel_spacing[0], base + folder + "centerline_init_id_"+str(muslce_ids[muscle_id])+".vtp", [0,0,0])


point_hor, normal_vector_hor = fit_plane_to_two_centerlines(extended_centerlines[pairs[0][0]], extended_centerlines[pairs[0][1]])
point_vert, normal_vector_vert = fit_plane_to_two_centerlines(extended_centerlines[pairs[1][0]], extended_centerlines[pairs[1][1]])

print("Check of angle  = ", numpy.dot(normal_vector_hor, normal_vector_vert))

# Plane 1 intersections
pts_plane_horizontal = find_plane_curve_intersections(section, section_curve, point_hor*pixel_spacing[0], normal_vector_hor)
# Plane 2 intersections
pts_plane_vertical = find_plane_curve_intersections(section, section_curve, point_vert*pixel_spacing[0], normal_vector_vert)


# Plane 1B intersections
pts_plane_B_horizontal = find_plane_curve_intersections(section_B, section_curve_B, point_hor*pixel_spacing[0], normal_vector_hor)
# Plane 2B intersections
pts_plane_B_vertical = find_plane_curve_intersections(section_B, section_curve_B, point_vert*pixel_spacing[0], normal_vector_vert)

for id in range(len(muslce_ids)):

    print(id)

    # # horizontal pair
    # if muscle_id == pairs[0][1]:
    #     target_1 = pts_plane_horizontal[0]/pixel_spacing[0]
    #     target_1_B = pts_plane_B_horizontal[1]/pixel_spacing[0]

    # elif muscle_id == pairs[0][0]:
    #     target_1 = pts_plane_horizontal[1]/pixel_spacing[0]
    #     target_1_B = pts_plane_B_horizontal[0]/pixel_spacing[0]

    # # vertical pair
    # if muscle_id == pairs[1][0]:
    #     target_1 = pts_plane_vertical[1]/pixel_spacing[0]
    #     target_1_B = pts_plane_B_vertical[0]/pixel_spacing[0]

    # elif muscle_id == pairs[1][1]:
    #     target_1 = pts_plane_vertical[0]/pixel_spacing[0]
    #     target_1_B = pts_plane_B_vertical[1]/pixel_spacing[0]


    if id in pairs[0]:
        print("muslce id ", muslce_ids[id],    "horizontal")

        targets_1 = pts_plane_horizontal/pixel_spacing[0]
        targets_1_B = pts_plane_B_horizontal/pixel_spacing[0]

    if id in pairs[1]:
        print("     muslce id ", muslce_ids[id],   " vertical")

        targets_1 = pts_plane_vertical/pixel_spacing[0]
        targets_1_B = pts_plane_B_vertical/pixel_spacing[0]

    # Extract coordinates of the current muscle mask
    mask = muscles[id]

    muscle_voxels = numpy.argwhere(mask) * pixel_spacing[0]
    tree = cKDTree(muscle_voxels)
    
    # Query distance to the chosen target points
    dist_to_target_1, _ = tree.query(targets_1 * pixel_spacing[0])
    dist_to_target_1_B, _ = tree.query(targets_1_B * pixel_spacing[0])
    
    print("        -> Distance to target_1: ", dist_to_target_1)
    print("        -> Distance to target_1_B: ", dist_to_target_1_B)

    target_1 = targets_1[numpy.argmin(dist_to_target_1)]
    target_1_B = targets_1_B[numpy.argmin(dist_to_target_1_B)]


    _, centerline_ext_2 = fit_analytical_centerline(mask, [target_1_B, target_1, target_point], ignored_nodes, order, n_bins=10, extension_factor=0.25)

    extended_centerlines_1.append(centerline_ext_2)


    export_centerline(centerline_ext_2, pixel_spacing[0], base + folder + "centerline_exttended_2_id_"+str(muslce_ids[id])+".vtp", [0,0,0])



Check of angle  =  0.044353772672241415
0
muslce id  2 horizontal
        -> Distance to target_1:  [24.92519628 11.81574433]
        -> Distance to target_1_B:  [ 5.42911956 23.46008625]
1
     muslce id  3  vertical
        -> Distance to target_1:  [25.72387949 13.00318921]
        -> Distance to target_1_B:  [ 6.26158156 25.1417403 ]
2
muslce id  4 horizontal
        -> Distance to target_1:  [ 2.12239923 22.82877236]
        -> Distance to target_1_B:  [26.61222575  0.35846666]
3
     muslce id  5  vertical
        -> Distance to target_1:  [ 8.68078155 27.60137682]
        -> Distance to target_1_B:  [29.58190323  0.88885115]


## Find intersection of the extended centerline and the eyeball


In [10]:
def find_eye_intersection_index(centerline_extended, mask_eye):
    """
    Returns the index along the extended centerline that intersects the eye.
    """
    for idx, point in enumerate(centerline_extended):
        z, y, x = numpy.round(point).astype(int)
        
        # Check bounds
        if 0 <= z < mask_eye.shape[0] and 0 <= y < mask_eye.shape[1] and 0 <= x < mask_eye.shape[2]:
            if mask_eye[z, y, x] == 1:
                return idx  
                
    return len(centerline_extended) - 1


# Part II

In [11]:
def get_local_frame(direction):
    """Calculates two vectors perpendicular to the spine direction."""
    # Pick a non-parallel vector to start the cross product
    vec = numpy.array([1, 0, 0]) if abs(direction[0]) < 0.9 else numpy.array([0, 1, 0])
    v1 = numpy.cross(direction, vec)
    v1 /= numpy.linalg.norm(v1)
    v2 = numpy.cross(direction, v1)
    return v1, v2

## Extract profiles

In [12]:
def extract_radial_profiles_from_mask(centerline, mask, pixel_spacing, n_theta=64):
    """
    Step 1: Casts fixed radiating angular rays outward from each centerline node 
    to measure the exact distance (radius) to the mask boundary.
    """
    voxel_coords = numpy.argwhere(mask)
    if len(voxel_coords) == 0:
        return numpy.full((len(centerline), n_theta), numpy.nan)
        
    tree = cKDTree(voxel_coords)
    n_nodes = len(centerline)
    
    # Grid to hold our raw radii: rows = slices, columns = fixed angles
    radii_grid = numpy.zeros((n_nodes, n_theta))
    
    # Generate the fixed, matching angular target steps once
    theta_steps = numpy.linspace(-numpy.pi, numpy.pi, n_theta, endpoint=False)
    
    for idx in range(n_nodes):
        p = centerline[idx]
        
        # Calculate local tracking frame
        direction = centerline[min(idx+1, n_nodes-1)] - centerline[max(idx-1, 0)]
        norm_dir = numpy.linalg.norm(direction)
        if norm_dir == 0:
            radii_grid[idx, :] = numpy.nan
            continue
        direction /= norm_dir
        v1, v2 = get_local_frame(direction)
        
        # Broad neighborhood check
        close_idx = tree.query_ball_point(p, r=45)
        if len(close_idx) < 10:
            radii_grid[idx, :] = numpy.nan # Outside the mask volume
            continue
            
        local_voxels = voxel_coords[close_idx] - p
        dist_plane = numpy.dot(local_voxels, direction)
        slab = numpy.abs(dist_plane) < 1.0
        local_voxels = local_voxels[slab]
        
        if len(local_voxels) < 10:
            radii_grid[idx, :] = numpy.nan
            continue
            
        # Project local voxels onto 2D plane
        x = local_voxels @ v1
        y = local_voxels @ v2
        
        # Calculate polar coordinates of the raw slice voxels
        vox_theta = numpy.arctan2(y, x)
        vox_r = numpy.sqrt(x**2 + y**2)
        
        # Map the voxel distances onto our fixed angular target steps
        for t_idx, target_theta in enumerate(theta_steps):
            # Find boundary voxels near this specific ray angle (e.g., within 6 degrees)
            angle_diff = numpy.abs(numpy.arctan2(numpy.sin(vox_theta - target_theta), numpy.cos(vox_theta - target_theta)))
            angular_slice_mask = angle_diff < (2*numpy.pi / n_theta)
            
            if numpy.any(angular_slice_mask):
                # The maximum radius along this ray represents the outer boundary wall
                radii_grid[idx, t_idx] = numpy.max(vox_r[angular_slice_mask])
            else:
                radii_grid[idx, t_idx] = numpy.nan
                
    return radii_grid



# def smooth_and_extrapolate_radii(radii_grid, n_output_extended, smoothing=15.0):
#     """
#     PASS 1: In-slice circular filtering to smooth perimeters.
#     PASS 2: Unconstrained 1st-order (linear) longitudinal spline extrapolation.
#     """
#     n_trimmed, n_theta = radii_grid.shape
#     smooth_profiles = numpy.zeros((n_output_extended, n_theta))
    
#     # =========================================================================
#     # PASS 1: IN-SLICE PERIMETER SMOOTHING & CLEANUP (Unchanged)
#     # =========================================================================
#     for idx in range(n_trimmed):
#         row_series = radii_grid[idx, :].copy()
#         if numpy.all(numpy.isnan(row_series)):
#             continue
            
#         nans = numpy.isnan(row_series)
#         if numpy.any(nans):
#             xp = numpy.where(~nans)[0]
#             fp = row_series[~nans]
#             row_series[nans] = numpy.interp(numpy.where(nans)[0], xp, fp, period=n_theta)
            
#         filtered_row = median_filter(row_series, size=5, mode='wrap')
#         radii_grid[idx, :] = uniform_filter1d(filtered_row, size=5, mode='wrap')

#     # =========================================================================
#     # PASS 2: UNCONSTRAINED LINEAR LONGITUDINAL SPLINE (k_order=1)
#     # =========================================================================
#     # The coordinate system of the real segmented input data rows (0 to N-1)
#     u_real = numpy.arange(n_trimmed)
    
#     # The coordinate system of the entire target extended centerline (0 to M-1)
#     u_evaluation = numpy.arange(n_output_extended)
    
#     for t_idx in range(n_theta):
#         column = radii_grid[:, t_idx]
#         valid_mask = ~numpy.isnan(column)
        
#         # Linear spline requires at least 3 points (k+2)
#         if numpy.sum(valid_mask) >= 3:
#             # Fit a raw linear spline (k=1) using only your real mask data points
#             # ext=0 allows it to predictively extrapolate down the remaining tail
#             rib_spline = UnivariateSpline(
#                 u_real[valid_mask], 
#                 column[valid_mask], 
#                 k=1, 
#                 s=smoothing, 
#                 ext=0
#             )
            
#             # Evaluate across the full extended shape up to length M
#             extrapolated_radii = rib_spline(u_evaluation)
            
#             # Simple floor clamp to protect mesh topology from flattening to 0
#             smooth_profiles[:, t_idx] = numpy.clip(extrapolated_radii, 0.1, None)
#         else:
#             # Fallback if a specific angular ray has no valid data
#             smooth_profiles[:, t_idx] = 1.0

#     # Smooth the linear transitions slightly down the longitudinal columns
#     for t_idx in range(n_theta):
#         smooth_profiles[:, t_idx] = uniform_filter1d(
#             smooth_profiles[:, t_idx], 
#             size=5, 
#             mode='nearest'
#         )
        
#     return smooth_profiles



def smooth_and_extrapolate_radii(radii_grid, n_output_extended, smoothing=15.0):
    """
    PASS 1: Longitudinal 1st-order (linear) spline extrapolation down each angle.
    PASS 2: In-slice cyclic wrapping filters to polish the 2D cross-sections.
    """
    n_trimmed, n_theta = radii_grid.shape
    
    # Initialize the output grid to fit the full extended target length
    smooth_profiles = numpy.zeros((n_output_extended, n_theta))

    # =========================================================================
    # PASS 1: LONGITUDINAL SMOOTHING & LINEAR EXTRAPOLATION (Across-Slice First)
    # =========================================================================
    u_real = numpy.arange(n_trimmed)
    u_evaluation = numpy.arange(n_output_extended)
    
    for t_idx in range(n_theta):
        column = radii_grid[:, t_idx]
        valid_mask = ~numpy.isnan(column)
        
        # Repair internal gaps within this specific angular column before fitting
        if numpy.any(valid_mask) and numpy.sum(valid_mask) >= 3:
            # Linear spline (k=1) to project raw trajectory straight into extension zone
            rib_spline = UnivariateSpline(
                u_real[valid_mask], 
                column[valid_mask], 
                k=1, 
                s=smoothing, 
                ext=0
            )
            extrapolated_radii = rib_spline(u_evaluation)
            smooth_profiles[:, t_idx] = numpy.clip(extrapolated_radii, 0.1, None)
        else:
            smooth_profiles[:, t_idx] = 1.0

    # # Apply a quick longitudinal filter to stabilize the tracking strings
    # for t_idx in range(n_theta):
    #     smooth_profiles[:, t_idx] = uniform_filter1d(
    #         smooth_profiles[:, t_idx], 
    #         size=5, 
    #         mode='nearest'
    #     )

    # =========================================================================
    # PASS 2: IN-SLICE PERIMETER SMOOTHING (In-Slice Second)
    # =========================================================================
    # Now that the full grid (up to n_output_extended) is populated, we loop
    # through EVERY slice and polish its circular contour perimeter.
    for idx in range(n_output_extended):
        row_series = smooth_profiles[idx, :].copy()
        
        # Failsafe check for empty rows
        if numpy.all(numpy.isnan(row_series)):
            continue
            
        # Clean up any leftover interpolation gaps around the loop
        nans = numpy.isnan(row_series)
        if numpy.any(nans):
            xp = numpy.where(~nans)[0]
            fp = row_series[~nans]
            row_series[nans] = numpy.interp(numpy.where(nans)[0], xp, fp, period=n_theta)
            
        # Cyclic wrapping filters erase any remaining jagged voxel steps
        filtered_row = median_filter(row_series, size=5, mode='wrap')
        smooth_profiles[idx, :] = uniform_filter1d(filtered_row, size=5, mode='wrap')
        
    return smooth_profiles


def smooth_and_extrapolate_radii_reversed(radii_grid, n_output_extended, smoothing=15.0):
    """
    PASS 1: Longitudinal 1st-order (linear) spline extrapolation down each angle.
    PASS 2: In-slice cyclic wrapping filters to polish the 2D cross-sections.
    """
    n_trimmed, n_theta = radii_grid.shape
    
    # Initialize the output grid to fit the full extended target length (M)
    smooth_profiles = numpy.zeros((n_output_extended, n_theta))

    # =========================================================================
    # PASS 1: LONGITUDINAL SMOOTHING & LINEAR EXTRAPOLATION (Across-Slice First)
    # =========================================================================
    # Coordinate system of the real data rows (0 to N-1)
    u_real = numpy.arange(n_trimmed)
    # Complete target evaluation span (0 to M-1)
    u_evaluation = numpy.arange(n_output_extended)
    
    for t_idx in range(n_theta):
        column = radii_grid[:, t_idx]
        valid_mask = ~numpy.isnan(column)
        
        # Linear spline requires at least 3 points (k+2)
        if numpy.any(valid_mask) and numpy.sum(valid_mask) >= 3:
            # Linear spline (k=1) to project raw trajectory straight into extension zone
            # ext=0 allows native linear trend extrapolation down the empty tail
            rib_spline = UnivariateSpline(
                u_real[valid_mask], 
                column[valid_mask], 
                k=1, 
                s=smoothing, 
                ext=0
            )
            extrapolated_radii = rib_spline(u_evaluation)
            smooth_profiles[:, t_idx] = numpy.clip(extrapolated_radii, 0.1, None)
        else:
            # Fallback if an entire angle strip is empty or missing data
            smooth_profiles[:, t_idx] = 1.0

    # Apply a longitudinal filter pass to stabilize the angular tracking strings
    for t_idx in range(n_theta):
        smooth_profiles[:, t_idx] = uniform_filter1d(
            smooth_profiles[:, t_idx], 
            size=5, 
            mode='nearest'
        )

    # =========================================================================
    # PASS 2: IN-SLICE PERIMETER SMOOTHING (In-Slice Second)
    # =========================================================================
    # Run through EVERY slice up to the full extended length to clean up perimeters
    for idx in range(n_output_extended):
        row_series = smooth_profiles[idx, :].copy()
        
        if numpy.all(numpy.isnan(row_series)):
            continue
            
        # Clean up any leftover gaps around the circular loop
        nans = numpy.isnan(row_series)
        if numpy.any(nans):
            xp = numpy.where(~nans)[0]
            fp = row_series[~nans]
            row_series[nans] = numpy.interp(numpy.where(nans)[0], xp, fp, period=n_theta)
            
        # Cyclic wrapping filters erase jagged voxel boundaries seamlessly
        filtered_row = median_filter(row_series, size=5, mode='wrap')
        smooth_profiles[idx, :] = uniform_filter1d(filtered_row, size=5, mode='wrap')
        
    return smooth_profiles



## Generate mesh

In [13]:
def generate_radial_mesh(
    centerline, 
    radii_grid, 
    pixel_spacing_val,
    insertion_idx=None,
    taper_start_len=12,
    taper_end_len=12
):

    print(centerline.shape)
    print(radii_grid.shape)

    
    n_sections, n_theta = radii_grid.shape

    if insertion_idx is None:
        insertion_idx = n_sections

    n_sections = min(n_sections, insertion_idx)
    vertices = []
    
    # Generate matching angle steps matching your profile extractor map orientation
    theta_steps = numpy.linspace(-numpy.pi, numpy.pi, n_theta, endpoint=False)
    
    # The absolute 3D boundary endpoints for the physical collapse
    start_point = centerline[0]
    end_point = centerline[n_sections - 1]
    
    # -------------------------------------------------------------------------
    # 1. Coordinate Construction with Biological End Tapering
    # -------------------------------------------------------------------------
    for i in range(n_sections):
        p_node = centerline[i]
        
        # Calculate local orientation vectors
        direction = centerline[min(i + 1, n_sections - 1)] - centerline[max(i - 1, 0)]
        norm_dir = numpy.linalg.norm(direction)
        direction = direction / norm_dir if norm_dir > 0 else numpy.array([1.0, 0.0, 0.0])
        v1, v2 = get_local_frame(direction)
        
        # Read the pre-smoothed radial strip across all angles for this slice
        r_slice = radii_grid[i, :].copy()
        r_slice = numpy.maximum(r_slice, 0.02)
        
        # =========================================================================
        # PROXIMAL TAPER (Muscle Origin / Start)
        # =========================================================================
        if i < taper_start_len:
            t = i / max(taper_start_len - 1, 1)   # Progress tracking factor: 0 → 1
            scale = t**2                          # Smooth quadratic biological blend
            
            p_node = (1.0 - t) * start_point + t * p_node
            r_slice = r_slice * scale
            
        # =========================================================================
        # DISTAL TAPER (Eye Wall Insertion / End)
        # =========================================================================
        if i >= n_sections - taper_end_len:
            t = (i - (n_sections - taper_end_len)) / max(taper_end_len - 1, 1)
            scale = (1.0 - t)**2                  # Inverted quadratic decay profile
            
            p_node = (1.0 - t) * p_node + t * end_point
            r_slice = r_slice * scale
            
        # =========================================================================
        # Slice Ring Geometry Mapping
        # =========================================================================
        # Construct the complete circular vertex array loop for this slice profile
        ring = (
            p_node[None, :]
            + r_slice[:, None] * numpy.cos(theta_steps)[:, None] * v1[None, :]
            + r_slice[:, None] * numpy.sin(theta_steps)[:, None] * v2[None, :]
        )
        vertices.append(ring)
        
    vertices = numpy.vstack(vertices)
    
    # -------------------------------------------------------------------------
    # 2. Quad-to-Triangle Surface Topology Mapping (Continuous Tube)
    # -------------------------------------------------------------------------
    faces = []
    for i in range(n_sections - 1):
        base0 = i * n_theta
        base1 = (i + 1) * n_theta
        
        for j in range(n_theta):
            j2 = (j + 1) % n_theta
            
            v00 = base0 + j
            v01 = base0 + j2
            v10 = base1 + j
            v11 = base1 + j2
            
            # Winding triangles to construct clean, outwards-facing normals
            faces.append([v00, v10, v11])
            faces.append([v00, v11, v01])
            
    # -------------------------------------------------------------------------
    # 3. Compile scaled Trimesh geometry asset
    # -------------------------------------------------------------------------
    mesh = trimesh.Trimesh(
        vertices=vertices * pixel_spacing_val,
        faces=numpy.asarray(faces),
        process=False
    )
    
    return mesh

## Meshin loop init

In [14]:
# n_theta_points = 50
# smoothing_factor = 15
# spline_order = 3

# for muscle_id in range(len(muslce_ids)):
#     print("Running Simplified Radial Pipeline on Muscle:", muslce_ids[muscle_id])


#     eye_intersection_idx = find_eye_intersection_index(
#     extended_centerlines_1[muscle_id],
#     mask_eye
#     )

#     short_centerline = centerlines[muscle_id]            # Original unextended path
#     full_centerline = extended_centerlines_1[muscle_id]  # Extended path going to eye
#     centerline_trimmed = extended_centerlines_1[muscle_id][:eye_intersection_idx+1]


#    # ------------------------------------------------------------------------
#     # 2. Step 1: Run the raw geometric radar sweep
#     # -------------------------------------------------------------------------
#     # This samples the mask data along the original short centerline length
#     radii_raw = extract_radial_profiles_from_mask(
#         centerline=short_centerline,
#         mask=muscles[muscle_id],
#         pixel_spacing=pixel_spacing,
#         n_theta=n_theta_points
#     )
    
#     # -------------------------------------------------------------------------
#     # 3. Steps 2 & 3: Smooth and Extrapolate to the Target Size
#     radii_processed = smooth_and_extrapolate_radii(
#         radii_grid=radii_raw,
#         n_output_extended=len(short_centerline),
#         smoothing=15.0  # Adjust 's' to make the linear trends stiffer or looser
#     )

#     # -------------------------------------------------------------------------
#     # 4. Step 4: Generate the surface mesh with biological end-tapers
#     # -------------------------------------------------------------------------
#     # We pass the full_centerline and set insertion_idx to match the eye boundary
#     mesh = generate_radial_mesh(
#         centerline=short_centerline,
#         radii_grid=radii_processed,
#         pixel_spacing_val=pixel_spacing[0],
#         insertion_idx=eye_intersection_idx + 1,  # Clips rendering at eye wall
#         taper_start_len=15,                      # Collapses origin smoothly
#         taper_end_len=0                         # Collapses insertion smoothly
#     )
    
#     mesh.export(export_name + "_RADIAL_id_" + str(muslce_ids[muscle_id]) + ".stl")

## Contact surface on the eyeball mesh

In [15]:
import pyvista as pv
def get_eyeball_contact_surface(eyeball_stl_path, centerline_extended, eye_intersection_idx, 
                                pixel_spacing, target_radius_mm=2.0, n_theta=50, 
                                output_vtk_path="eyeball_contact.vtk"):
    """
    1. Identifies the contact patch on the eyeball STL.
    2. Uses a pure-Python dictionary walker to trace the exact outer boundary loop.
    3. Resamples the rim uniformly to precisely n_theta points.
    """
    # Load the eyeball mesh
    mesh_trimesh = trimesh.load(eyeball_stl_path)
    vertices = mesh_trimesh.vertices
    faces = mesh_trimesh.faces
    
    # Calculate tracking math in mm space
    contact_voxel_coord = centerline_extended[eye_intersection_idx]
    contact_point_mm = contact_voxel_coord * pixel_spacing
    
    # Find the exact touch center node
    mesh_tree = cKDTree(vertices)
    _, closest_vertex_idx = mesh_tree.query(contact_point_mm)
    mesh_contact_point = vertices[closest_vertex_idx]
    
    # -------------------------------------------------------------------------
    # 1. Extract the Contact Disc Sub-Mesh Patch
    # -------------------------------------------------------------------------
    face_centers = mesh_trimesh.triangles_center
    face_tree = cKDTree(face_centers)
    patch_face_indices = face_tree.query_ball_point(mesh_contact_point, r=target_radius_mm)
    
    if len(patch_face_indices) < 3:
        raise ValueError("Target radius is too small; no mesh faces captured.")
        
    patch_mesh = mesh_trimesh.submesh([patch_face_indices], append=True)
    
    # -------------------------------------------------------------------------
    # 2. Extract Boundary Edges via Unique Grouping
    # -------------------------------------------------------------------------
    edges_sorted = patch_mesh.edges_sorted
    unique_edge_indices = trimesh.grouping.group_rows(edges_sorted, require_count=1)
    boundary_edges = patch_mesh.edges[unique_edge_indices]
    
    # -------------------------------------------------------------------------
    # 3. PURE PYTHON ADJACENCY WALK (Version-Agnostic Loop Sorter)
    # -------------------------------------------------------------------------
    # Map every vertex on the rim to its two immediate neighbors
    adjacency = {}
    for u, v in boundary_edges:
        adjacency.setdefault(u, []).append(v)
        adjacency.setdefault(v, []).append(u)
        
    # Start at an arbitrary edge segment node and follow the chain around the rim
    start_node = boundary_edges[0][0]
    ordered_patch_vertex_indices = [start_node]
    
    # Move to the first connected neighbor
    current_node = adjacency[start_node][0]
    prev_node = start_node
    
    # Step sequentially from edge to edge until we loop back to the start index
    while current_node != start_node:
        ordered_patch_vertex_indices.append(current_node)
        neighbors = adjacency[current_node]
        
        # Pick the neighbor we didn't just come from
        next_node = neighbors[0] if neighbors[0] != prev_node else neighbors[1]
        
        prev_node = current_node
        current_node = next_node
        
    # Map back to absolute physical submesh vertex coordinates (mm)
    raw_rim_vertices = patch_mesh.vertices[ordered_patch_vertex_indices]
    
    # -------------------------------------------------------------------------
    # 4. Angular Synchronization & Resampling
    # -------------------------------------------------------------------------
    idx_next = min(eye_intersection_idx + 1, len(centerline_extended) - 1)
    idx_prev = max(eye_intersection_idx - 1, 0)
    tangent = (centerline_extended[idx_next] - centerline_extended[idx_prev]) * pixel_spacing
    tangent /= numpy.linalg.norm(tangent)
    
    if numpy.abs(tangent[0]) < 0.9:
        v1 = numpy.cross(tangent, [1, 0, 0])
    else:
        v1 = numpy.cross(tangent, [0, 1, 0])
    v1 /= numpy.linalg.norm(v1)
    v2 = numpy.cross(tangent, v1)
    v2 /= numpy.linalg.norm(v2)
    
    # Project raw rim vertices to find robust polar angles
    rel_rim = raw_rim_vertices - mesh_contact_point
    rim_angles = numpy.arctan2(rel_rim @ v2, rel_rim @ v1)
    
    # Sort monotonically for a clean numpy.interp pass
    sort_idx = numpy.argsort(rim_angles)
    sorted_rim_angles = rim_angles[sort_idx]
    sorted_rim_vertices = raw_rim_vertices[sort_idx]
    
    # Interpolate x, y, z coordinates uniformly over the target_angles
    target_angles = numpy.linspace(-numpy.pi, numpy.pi, n_theta, endpoint=False)
    ordered_contour_vertices = numpy.zeros((n_theta, 3))
    
    for i in range(3):
        ordered_contour_vertices[:, i] = numpy.interp(
            target_angles, sorted_rim_angles, sorted_rim_vertices[:, i], period=2*numpy.pi
        )

    # -------------------------------------------------------------------------
    # 5. Export Attributed VTK Data
    # -------------------------------------------------------------------------
    nv = faces.shape[0]
    padding = numpy.ones((nv, 1), dtype=int) * 3
    pv_faces = numpy.hstack((padding, faces)).ravel()
    grid = pv.PolyData(vertices, pv_faces)
    
    # Map our clean resampled contour back to global mesh points for visualization
    global_rim_indices = []
    global_tree = cKDTree(vertices)
    for pt in ordered_contour_vertices:
        _, glob_idx = global_tree.query(pt)
        global_rim_indices.append(glob_idx)
        
    global_patch_v_indices = numpy.unique(faces[patch_face_indices].ravel())
    
    node_registration = numpy.zeros(grid.n_points, dtype=numpy.int32)
    node_registration[global_patch_v_indices] = 2  # The solid cluster disc
    node_registration[global_rim_indices] = 3      # The clean outer boundary ring
    node_registration[closest_vertex_idx] = 1     # The touch center node
    
    grid.point_data["NodeRegistration"] = node_registration
    grid.save(output_vtk_path)
    
    print(f"--> [SUCCESS] Saved VTK diagnostic mesh to: {output_vtk_path}")
    return mesh_contact_point, ordered_contour_vertices


## Set meshing parameters

In [16]:
n_theta_points = 50
smoothing_factor = 15
spline_order = 3

## Get contact points & surfaces

In [17]:
contact_centers = []
eye_surfaces = []
eye_intersection_IDs = []

for muscle_id in range(len(muslce_ids)):
    # ... Locate eye intersection index ...
    
    eyeball_stl = f"{export_name}_surf_id_{str(eyeball_id)}_outer.stl"
    output_vtk = f"{export_name}_muscle_{muscle_id}_eye_contact.vtk"

    eye_intersection_idx = find_eye_intersection_index(
    extended_centerlines_1[muscle_id],
    mask_eye
    )
    eye_intersection_IDs.append(eye_intersection_idx)
    
    # Call updated function: writes VTK and passes back the raw coordinate metrics
    contact_center, eye_wall_ring = get_eyeball_contact_surface(
        eyeball_stl_path=eyeball_stl,
        centerline_extended=extended_centerlines_1[muscle_id],
        eye_intersection_idx=eye_intersection_idx,
        pixel_spacing=pixel_spacing,
        target_radius_mm=2.0, # Target width
        n_theta=n_theta_points,
        output_vtk_path=output_vtk
    )

    contact_centers.append(contact_center)
    eye_surfaces.append(eye_wall_ring)

--> [SUCCESS] Saved VTK diagnostic mesh to: /Users/skardova/Documents/MRI_data/Brain/Comprehensive ultrahigh resolution/ses-4712/Radial-profiles-from-mask/Segmentation_1._muscle_0_eye_contact.vtk
--> [SUCCESS] Saved VTK diagnostic mesh to: /Users/skardova/Documents/MRI_data/Brain/Comprehensive ultrahigh resolution/ses-4712/Radial-profiles-from-mask/Segmentation_1._muscle_1_eye_contact.vtk
--> [SUCCESS] Saved VTK diagnostic mesh to: /Users/skardova/Documents/MRI_data/Brain/Comprehensive ultrahigh resolution/ses-4712/Radial-profiles-from-mask/Segmentation_1._muscle_2_eye_contact.vtk
--> [SUCCESS] Saved VTK diagnostic mesh to: /Users/skardova/Documents/MRI_data/Brain/Comprehensive ultrahigh resolution/ses-4712/Radial-profiles-from-mask/Segmentation_1._muscle_3_eye_contact.vtk


## Meshing function

In [ ]:
import numpy as np
import trimesh

def generate_blended_mesh(full_centerline, short_radii_grid, eye_wall_ring, contact_center_mm, pixel_spacing_val, origin_center_mm=None):
    """
    Generates a simulation-ready muscle mesh optimized for FEM solvers.
    Uses an elliptical dome (bullet-cap) profile at the origin to eliminate 
    sharp geometric singularities and prevent squashed sliver elements.
    """
    n_total_slices = len(full_centerline)             
    n_real_slices, n_theta = short_radii_grid.shape    
    vertices = []
    
    theta_steps = numpy.linspace(-numpy.pi, numpy.pi, n_theta, endpoint=False)
    
    # 1. Coordinate conversion
    eye_wall_ring_vox = eye_wall_ring / pixel_spacing_val
    contact_center_vox = contact_center_mm / pixel_spacing_val
    
    if origin_center_mm is None:
        origin_center_vox = full_centerline[0] - (full_centerline[1] - full_centerline[0]) * 1.0
    else:
        origin_center_vox = origin_center_mm / pixel_spacing_val
    
    # 2. Extract eyeball target local profile
    end_node = full_centerline[-1]
    dir_end = full_centerline[-1] - full_centerline[-2]
    dir_end /= numpy.linalg.norm(dir_end)
    v1_end, v2_end = get_local_frame(dir_end)
    
    relative_eye_pts = eye_wall_ring_vox - end_node
    eye_local_x = relative_eye_pts @ v1_end
    eye_local_y = relative_eye_pts @ v2_end
    
    # 3. Build geometry slice-by-slice
    for i in range(n_total_slices):
        p_node = full_centerline[i]
        
        direction = full_centerline[min(i + 1, n_total_slices - 1)] - full_centerline[max(i - 1, 0)]
        norm_dir = numpy.linalg.norm(direction)
        direction = direction / norm_dir if norm_dir > 0 else numpy.array([1.0, 0.0, 0.0])
        v1, v2 = get_local_frame(direction)
        
        # --- ZONE A: THE REAL SEGMENTED MUSCLE BODY ---
        if i < n_real_slices:
            r_slice = short_radii_grid[i, :].copy()
            
            # =========================================================================
            # ELLIPTICAL DOME CAP (Proximal End)
            # =========================================================================
            taper_len = min(12, n_real_slices // 4)
            if i < taper_len:
                # Normalized progress from 0 (tip) to 1 (full muscle body)
                t = i / max(taper_len - 1, 1)
                
                # Hemispherical/Elliptical profile arc formula
                scale = numpy.sqrt(t * (2.0 - t))  
                
                r_slice = r_slice * scale
                # Smoothly transition the centerline position to the apex point
                p_node = (1.0 - t) * origin_center_vox + t * p_node

            r_slice = numpy.maximum(r_slice, 0.05) # Keep a tiny healthy minimum floor for tet-meshing
            
            ring = (
                p_node[None, :]
                + r_slice[:, None] * numpy.cos(theta_steps)[:, None] * v1[None, :]
                + r_slice[:, None] * numpy.sin(theta_steps)[:, None] * v2[None, :]
            )
            vertices.append(ring)
            
        # --- ZONE B: THE TRANSITIONAL TUBE ---
        elif i < n_total_slices - 1:
            gap_total_steps = (n_total_slices - 1) - n_real_slices
            current_gap_step = i - n_real_slices + 1
            t = current_gap_step / max(gap_total_steps, 1)
            
            r_tail = short_radii_grid[-1, :]
            tail_local_x = r_tail * numpy.cos(theta_steps)
            tail_local_y = r_tail * numpy.sin(theta_steps)
            
            blended_local_x = (1.0 - t) * tail_local_x + t * eye_local_x
            blended_local_y = (1.0 - t) * tail_local_y + t * eye_local_y
            
            ring = (
                p_node[None, :]
                + blended_local_x[:, None] * v1[None, :]
                + blended_local_y[:, None] * v2[None, :]
            )
            vertices.append(ring)
            
        # --- ZONE C: THE FINAL SEAM ---
        else:
            vertices.append(eye_wall_ring_vox)
        
    vertices = numpy.vstack(vertices)
    
    # 4. Append tip vertices
    start_tip_idx = len(vertices)
    vertices = numpy.vstack([vertices, origin_center_vox[None, :]])
    
    end_tip_idx = len(vertices)
    vertices = numpy.vstack([vertices, contact_center_vox[None, :]])
    
    # 5. Construct Faces Matrix
    faces = []
    
    # Proximal Cap Triangles
    for j in range(n_theta):
        j2 = (j + 1) % n_theta
        faces.append([start_tip_idx, j2, j])
        
    # Main tube body quads
    for i in range(n_total_slices - 1):
        base0 = i * n_theta
        base1 = (i + 1) * n_theta
        for j in range(n_theta):
            j2 = (j + 1) % n_theta
            faces.append([base0 + j, base1 + j, base1 + j2])
            faces.append([base0 + j, base1 + j2, base0 + j2])
            
    # Distal Cap Triangles
    final_ring_base = (n_total_slices - 1) * n_theta
    for j in range(n_theta):
        j2 = (j + 1) % n_theta
        faces.append([final_ring_base + j, end_tip_idx, final_ring_base + j2])
        
    mesh = trimesh.Trimesh(
        vertices=vertices * pixel_spacing_val,
        faces=numpy.asarray(faces),
        process=False
    )
    
    return mesh


def generate_pure_blended_mesh(full_centerline, short_radii_grid, eye_wall_ring, contact_center_mm, pixel_spacing_val):
    """
    Generates a muscle mesh that natively handles sharp centerline twists.
    Morphs cross-sections inside their local 2D tracking frames before 
    projecting into 3D space, preventing self-intersections and thinning.
    """
    n_total_slices = len(full_centerline)             # Total path points (M)
    n_real_slices, n_theta = short_radii_grid.shape    # Real mask slices (N)
    vertices = []
    
    theta_steps = numpy.linspace(-numpy.pi, numpy.pi, n_theta, endpoint=False)
    
    # 1. Convert absolute millimeter inputs back to voxel space
    eye_wall_ring_vox = eye_wall_ring / pixel_spacing_val
    contact_center_vox = contact_center_mm / pixel_spacing_val
    
    # 2. Extract the local 2D profile of the eyeball target surface
    # Project the eye ring relative to the final centerline node's tracking frame
    end_node = full_centerline[-1]
    dir_end = full_centerline[-1] - full_centerline[-2]
    dir_end /= numpy.linalg.norm(dir_end)
    v1_end, v2_end = get_local_frame(dir_end)
    
    relative_eye_pts = eye_wall_ring_vox - end_node
    eye_local_x = relative_eye_pts @ v1_end
    eye_local_y = relative_eye_pts @ v2_end
    
    # 3. Build the combined geometry slice-by-slice
    for i in range(n_total_slices):
        p_node = full_centerline[i]
        
        # Calculate local tracking orientation vectors for the CURRENT node
        direction = full_centerline[min(i + 1, n_total_slices - 1)] - full_centerline[max(i - 1, 0)]
        norm_dir = numpy.linalg.norm(direction)
        direction = direction / norm_dir if norm_dir > 0 else numpy.array([1.0, 0.0, 0.0])
        v1, v2 = get_local_frame(direction)
        
        # --- ZONE A: THE REAL SEGMENTED MUSCLE BODY (Slices 0 to N-1) ---
        if i < n_real_slices:
            r_slice = short_radii_grid[i, :].copy()
            r_slice = numpy.maximum(r_slice, 0.02)
            
            # Pure local frame geometry generation
            ring = (
                p_node[None, :]
                + r_slice[:, None] * numpy.cos(theta_steps)[:, None] * v1[None, :]
                + r_slice[:, None] * numpy.sin(theta_steps)[:, None] * v2[None, :]
            )
            vertices.append(ring)
            
        # --- ZONE B: THE TRANSITIONAL TUBE (Slices N to M-2) ---
        elif i < n_total_slices - 1:
            gap_total_steps = (n_total_slices - 1) - n_real_slices
            current_gap_step = i - n_real_slices + 1
            t = current_gap_step / max(gap_total_steps, 1)
            
            # Extract the muscle tail's raw 2D representation
            r_tail = short_radii_grid[-1, :]
            tail_local_x = r_tail * numpy.cos(theta_steps)
            tail_local_y = r_tail * numpy.sin(theta_steps)
            
            # Linearly blend the local 2D coordinates
            # This morphs the shape layout cleanly before rotation is applied
            blended_local_x = (1.0 - t) * tail_local_x + t * eye_local_x
            blended_local_y = (1.0 - t) * tail_local_y + t * eye_local_y
            
            # Inflate the 2D blended profile using the current slice's orientation vectors
            ring = (
                p_node[None, :]
                + blended_local_x[:, None] * v1[None, :]
                + blended_local_y[:, None] * v2[None, :]
            )
            vertices.append(ring)
            
        # --- ZONE C: THE FINAL SEAM (Slice M-1) ---
        else:
            # Re-verify that the terminal ring matches raw un-approximated eye coordinates
            vertices.append(eye_wall_ring_vox)
        
    vertices = numpy.vstack(vertices)
    
    # 4. Append the center tip vertex to close the cap geometry
    tip_vertex_idx = len(vertices)
    vertices = numpy.vstack([vertices, contact_center_vox[None, :]])
    
    # 5. Construct Faces Matrix
    faces = []
    for i in range(n_total_slices - 1):
        base0 = i * n_theta
        base1 = (i + 1) * n_theta
        for j in range(n_theta):
            j2 = (j + 1) % n_theta
            v00 = base0 + j
            v01 = base0 + j2
            v10 = base1 + j
            v11 = base1 + j2
            faces.append([v00, v10, v11])
            faces.append([v00, v11, v01])
            
    # Triangulate the final ring to the tip vertex
    final_ring_base = (n_total_slices - 1) * n_theta
    for j in range(n_theta):
        j2 = (j + 1) % n_theta
        v_current = final_ring_base + j
        v_next = final_ring_base + j2
        faces.append([v_current, tip_vertex_idx, v_next])
        
    # Scale and export to millimeter space
    mesh = trimesh.Trimesh(
        vertices=vertices * pixel_spacing_val,
        faces=numpy.asarray(faces),
        process=False
    )
    
    return mesh

## Meshing loop - 2

In [19]:
blend_window_slices = 10  # Smooth transition run-up zone inside the muscle body

for muscle_id in range(len(muslce_ids)):
    print(f"Processing Unpolluted Asset Pipeline for Muscle: {muslce_ids[muscle_id]}")

    eye_intersection_idx = eye_intersection_IDs[muscle_id]
    short_centerline = centerlines[muscle_id] # Length N
    
    # Trim extended path exactly to the eye surface touchdown (Length M)
    centerline_trimmed = extended_centerlines_1[muscle_id][:eye_intersection_idx + 1]
    target_eye_wall_ring = eye_surfaces[muscle_id]
    contact_center = contact_centers[muscle_id]

    # Step 1: Sweep raw cross-sections ONLY along the real segmented short path (N)
    radii_raw = extract_radial_profiles_from_mask(
        centerline=short_centerline,
        mask=muscles[muscle_id],
        pixel_spacing=pixel_spacing,
        n_theta=n_theta_points
    )
    

    radii_pure_smoothed = smooth_and_extrapolate_radii(
        radii_grid=radii_raw,
        n_output_extended=len(short_centerline),
        smoothing=15.0  # Adjust 's' to make the linear trends stiffer or looser
    )


    # Step 3: Generate the mesh by bridging N data directly to the M target mesh ring
    # Compile the closed mesh
    mesh = generate_blended_mesh(
        full_centerline=centerline_trimmed,    
        short_radii_grid=radii_pure_smoothed,  
        eye_wall_ring=target_eye_wall_ring,    
        contact_center_mm=contact_center,      # Pass the absolute millimeter center point here
        pixel_spacing_val=pixel_spacing[0]
    )

    
    # Export the clean, artifact-free muscle asset
    output_filename = f"{export_name}_BLENDED_id_{muslce_ids[muscle_id]}.stl"
    mesh.export(output_filename)
    print(f"Successfully generated clean mesh: {output_filename}\n")


Processing Unpolluted Asset Pipeline for Muscle: 2
Successfully generated clean mesh: /Users/skardova/Documents/MRI_data/Brain/Comprehensive ultrahigh resolution/ses-4712/Radial-profiles-from-mask/Segmentation_1._BLENDED_id_2.stl

Processing Unpolluted Asset Pipeline for Muscle: 3
Successfully generated clean mesh: /Users/skardova/Documents/MRI_data/Brain/Comprehensive ultrahigh resolution/ses-4712/Radial-profiles-from-mask/Segmentation_1._BLENDED_id_3.stl

Processing Unpolluted Asset Pipeline for Muscle: 4
Successfully generated clean mesh: /Users/skardova/Documents/MRI_data/Brain/Comprehensive ultrahigh resolution/ses-4712/Radial-profiles-from-mask/Segmentation_1._BLENDED_id_4.stl

Processing Unpolluted Asset Pipeline for Muscle: 5
Successfully generated clean mesh: /Users/skardova/Documents/MRI_data/Brain/Comprehensive ultrahigh resolution/ses-4712/Radial-profiles-from-mask/Segmentation_1._BLENDED_id_5.stl



# Visualisation & Testing

In [20]:

# def debug_filtered_radial_slice(centerline, mask, pixel_spacing, slice_idx, n_theta=64, filter_size=5):
#     """
#     Visualizes a single 2D slice with an interactive option to test cyclic 
#     intraslice filtering (median + mean) to eliminate boundary spikes.
#     """
#     voxel_coords = numpy.argwhere(mask)
#     if len(voxel_coords) == 0:
#         print("Mask is completely empty.")
#         return
        
#     p = centerline[slice_idx]
    
#     # 1. Local tracking frame setup
#     direction = centerline[min(slice_idx+1, len(centerline)-1)] - centerline[max(slice_idx-1, 0)]
#     norm_dir = numpy.linalg.norm(direction)
#     if norm_dir == 0:
#         print(f"Slice {slice_idx} has a zero tangent vector.")
#         return
#     direction /= norm_dir
#     v1, v2 = get_local_frame(direction)
    
#     # 2. Extract neighbor plane voxels (+/- 1.0 voxel slab)
#     distances = numpy.linalg.norm(voxel_coords - p, axis=1)
#     close_mask = distances < 45
#     local_voxels = voxel_coords[close_mask] - p
    
#     dist_plane = numpy.dot(local_voxels, direction)
#     slab = numpy.abs(dist_plane) < 1.0
#     local_voxels = local_voxels[slab]
    
#     if len(local_voxels) < 5:
#         print(f"Slice {slice_idx} does not intersect enough mask voxels.")
#         return
        
#     x = local_voxels @ v1
#     y = local_voxels @ v2
    
#     vox_theta = numpy.arctan2(y, x)
#     vox_r = numpy.sqrt(x**2 + y**2)
    
#     # 3. Extract Raw Radar Sweep Radii
#     theta_steps = numpy.linspace(-numpy.pi, numpy.pi, n_theta, endpoint=False)
#     raw_radii = numpy.zeros(n_theta)
    
#     for t_idx, target_theta in enumerate(theta_steps):
#         angle_diff = numpy.abs(numpy.arctan2(numpy.sin(vox_theta - target_theta), numpy.cos(vox_theta - target_theta)))
#         angular_slice_mask = angle_diff < (numpy.pi / 30)
        
#         if numpy.any(angular_slice_mask):
#             raw_radii[t_idx] = numpy.max(vox_r[angular_slice_mask])
#         else:
#             raw_radii[t_idx] = numpy.nan
            
#     # Clean up isolated missing rays using linear interpolation across the loop
#     nans = numpy.isnan(raw_radii)
#     if numpy.any(nans):
#         # Handle wrapping boundary interpolation safely
#         xp = numpy.where(~nans)[0]
#         fp = raw_radii[~nans]
#         raw_radii[nans] = numpy.interp(numpy.where(nans)[0], xp, fp, period=n_theta)

#     # -------------------------------------------------------------------------
#     # STEP 4: APPLY THE CYCLIC FILTER
#     # -------------------------------------------------------------------------
#     smooth_radii = median_filter(raw_radii, size=filter_size, mode='wrap')
#     smooth_radii = uniform_filter1d(smooth_radii, size=filter_size, mode='wrap')

#     # Convert both raw and smoothed profiles back to 2D coordinates for plotting
#     raw_x = raw_radii * numpy.cos(theta_steps)
#     raw_y = raw_radii * numpy.sin(theta_steps)
#     raw_x = numpy.append(raw_x, raw_x[0]) # close loop
#     raw_y = numpy.append(raw_y, raw_y[0])

#     smooth_x = smooth_radii * numpy.cos(theta_steps)
#     smooth_y = smooth_radii * numpy.sin(theta_steps)
#     smooth_x = numpy.append(smooth_x, smooth_x[0]) # close loop
#     smooth_y = numpy.append(smooth_y, smooth_y[0])

#     # -------------------------------------------------------------------------
#     # 5. Plotting Comparison
#     # -------------------------------------------------------------------------
#     plt.figure(figsize=(9, 9))
    
#     # Draw raw voxel matrix background
#     plt.scatter(x, y, color='royalblue', s=45, marker='s', alpha=0.3, label='Raw Mask Pixels')
#     plt.plot(0, 0, 'go', markersize=10, zorder=5, label='Centerline Center (0,0)')
    
#     # Draw the spiked raw footprint
#     plt.plot(raw_x, raw_y, color='orange', linewidth=1.2, linestyle='--', label='Raw Spiked Footprint')
    
#     # Draw the beautifully continuous filtered boundary path
#     plt.plot(smooth_x, smooth_y, color='crimson', linewidth=2.5, label=f'Filtered Contour (size={filter_size})')
#     plt.scatter(smooth_x[:-1], smooth_y[:-1], color='crimson', s=20, zorder=4) # Final vertex dots
    
#     plt.axhline(0, color='black', linewidth=0.5, linestyle='--')
#     plt.axvline(0, color='black', linewidth=0.5, linestyle='--')
#     plt.axis('equal')
#     plt.title(f"Intraslice Filtering Check — Muscle Slice Index: {slice_idx}")
#     plt.xlabel("Local X (v1)")
#     plt.ylabel("Local Y (v2)")
#     plt.legend(loc='upper right')
#     plt.grid(True, linestyle=':', alpha=0.3)
#     plt.show()


# filter_size=5

# target_id = 3
# total_slices = len(centerlines[target_id])

# # Run it on the exact slice that was warping your 3D mesh
# debug_filtered_radial_slice(
#     centerlines[target_id], 
#     muscles[target_id], 
#     pixel_spacing, 
#     slice_idx=total_slices // 2, 
#     n_theta=64, 
#     filter_size=filter_size
# )




In [21]:
# def debug_global_mask_median_vs_angular(centerline, raw_mask, pre_filtered_mask, pixel_spacing, slice_idx, n_theta=64, filter_size=5):
#     """
#     Compares:
#     Left: Raw mask sliced -> angular cyclic 'spider' filter applied to the radii.
#     Right: Globally pre-filtered mask sliced -> raw radar sweep (no further filtering).
#     """
#     p = centerline[slice_idx]
    
#     # 1. Setup Local Tracking Frame
#     direction = centerline[min(slice_idx+1, len(centerline)-1)] - centerline[max(slice_idx-1, 0)]
#     norm_dir = numpy.linalg.norm(direction)
#     if norm_dir == 0:
#         print(f"Slice {slice_idx} has a zero tangent vector.")
#         return
#     direction /= norm_dir
#     v1, v2 = get_local_frame(direction)
    
#     # 2. Helper to extract local slice coordinates from a given mask
#     def get_slice_coordinates(mask_volume):
#         voxel_coords = numpy.argwhere(mask_volume)
#         if len(voxel_coords) == 0:
#             return numpy.array([]), numpy.array([])
        
#         distances = numpy.linalg.norm(voxel_coords - p, axis=1)
#         close_mask = distances < 45
#         local_voxels = voxel_coords[close_mask] - p
        
#         dist_plane = numpy.dot(local_voxels, direction)
#         slab = numpy.abs(dist_plane) < 1.0
#         local_voxels = local_voxels[slab]
        
#         if len(local_voxels) < 5:
#             return numpy.array([]), numpy.array([])
            
#         px = local_voxels @ v1
#         py = local_voxels @ v2
#         return px, py

#     # 3. Helper to extract the radar footprint
#     def extract_footprint(px, py, theta_steps):
#         if len(px) == 0:
#             return numpy.zeros(len(theta_steps))
#         v_theta = numpy.arctan2(py, px)
#         v_r = numpy.sqrt(px**2 + py**2)
#         radii = numpy.zeros(len(theta_steps))
        
#         for t_idx, target_theta in enumerate(theta_steps):
#             angle_diff = numpy.abs(numpy.arctan2(numpy.sin(v_theta - target_theta), numpy.cos(v_theta - target_theta)))
#             angular_slice_mask = angle_diff < (numpy.pi / 30)
#             if numpy.any(angular_slice_mask):
#                 radii[t_idx] = numpy.max(v_r[angular_slice_mask])
#             else:
#                 radii[t_idx] = numpy.nan
                
#         nans = numpy.isnan(radii)
#         if numpy.any(nans) and not numpy.all(nans):
#             xp = numpy.where(~nans)[0]
#             fp = radii[~nans]
#             radii[nans] = numpy.interp(numpy.where(nans)[0], xp, fp, period=len(theta_steps))
#         elif numpy.all(nans):
#             radii[:] = 1.0
#         return radii

#     theta_steps = numpy.linspace(-numpy.pi, numpy.pi, n_theta, endpoint=False)
    
#     # Extract coordinates for both masks on this plane
#     raw_x, raw_y = get_slice_coordinates(raw_mask)
#     filtered_x, filtered_y = get_slice_coordinates(pre_filtered_mask)
    
#     if len(raw_x) == 0 or len(filtered_x) == 0:
#         print(f"Slice {slice_idx} lacks enough overlapping voxels in one of the masks.")
#         return

#     # --- Left Plot Line: Your Angular Flow ---
#     raw_radii = extract_footprint(raw_x, raw_y, theta_steps)
#     smooth_radii = median_filter(raw_radii, size=filter_size, mode='wrap')
#     smooth_radii = uniform_filter1d(smooth_radii, size=filter_size, mode='wrap')

#     # --- Right Plot Line: Raw Sweep of the Pre-Filtered Mask ---
#     global_filtered_radii = extract_footprint(filtered_x, filtered_y, theta_steps)

#     # Prepare closed loops for rendering
#     def close_loop(r):
#         cx = r * numpy.cos(theta_steps)
#         cy = r * numpy.sin(theta_steps)
#         return numpy.append(cx, cx[0]), numpy.append(cy, cy[0])

#     raw_x_loop, raw_y_loop = close_loop(raw_radii)
#     smooth_x_loop, smooth_y_loop = close_loop(smooth_radii)
#     global_f_x_loop, global_f_y_loop = close_loop(global_filtered_radii)

#     # -------------------------------------------------------------------------
#     # PLOT RENDERING
#     # -------------------------------------------------------------------------
#     fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 8), sharex=True, sharey=True)
    
#     # --- LEFT PANEL: YOUR ANGULAR PIPELINE ---
#     ax1.scatter(raw_x, raw_y, color='royalblue', s=45, marker='s', alpha=0.3, label='Raw Mask Pixels')
#     ax1.plot(0, 0, 'go', markersize=10, zorder=5)
#     ax1.plot(raw_x_loop, raw_y_loop, color='orange', linewidth=1.2, linestyle='--', label='Raw Spiked Sweep')
#     ax1.plot(smooth_x_loop, smooth_y_loop, color='crimson', linewidth=2.5, label='Angular "Wrap" Filter')
#     ax1.scatter(smooth_x_loop[:-1], smooth_y_loop[:-1], color='crimson', s=15, zorder=4)
#     ax1.set_title("Your Method: Post-Extraction Angular Filtering\n(Sub-voxel precision, smooth continuity)", fontsize=11)
#     ax1.grid(True, linestyle=':', alpha=0.4)
#     ax1.legend(loc='upper right')
#     ax1.set_xlabel("Local X (v1)")
#     ax1.set_ylabel("Local Y (v2)")
#     # ax1.axis('equal')

#     # --- RIGHT PANEL: PRE-FILTERED MASK ---
#     ax2.scatter(raw_x, raw_y, color='royalblue', s=45, marker='s', alpha=0.3, label='Raw Mask Pixels')
#     ax2.scatter(filtered_x, filtered_y, color='forestgreen', s=45, marker='s', alpha=0.4, label='Pre-Filtered Mask Pixels')
#     ax2.plot(0, 0, 'go', markersize=10, zorder=5)
#     ax2.plot(global_f_x_loop, global_f_y_loop, color='darkgreen', linewidth=2.5, label='Raw Sweep Footprint')
#     ax2.scatter(global_f_x_loop[:-1], global_f_y_loop[:-1], color='darkgreen', s=15, zorder=4)
#     ax2.set_title(f"Alternative: Global Mask Median Filter (size={filter_size}) applied first\n(Traces original voxel blockiness directly)", fontsize=11)
#     ax2.grid(True, linestyle=':', alpha=0.4)
#     ax2.legend(loc='upper right')
#     ax2.set_xlabel("Local X (v1)")

#     plt.suptitle(f"Anatomical Preservation Check — Muscle Slice Index: {slice_idx}", fontsize=14, weight='bold')
#     plt.tight_layout()
#     plt.show()



# # DEFINE CONFIGURATION CONSTANTS
# # -------------------------------------------------------------------------
# filter_size = 5     # Size of the structural footprint kernel
# n_theta_points = 64 # Angular ray density
# target_id = 3       # The index of the specific extraocular muscle asset you are testing

# # Calculate explicit tracking index markers
# total_slices = len(centerlines[target_id])
# quarter_slice = total_slices // 4
# mid_slice     = total_slices // 2
# three_quarter = (3 * total_slices) // 4

# # -------------------------------------------------------------------------
# # RUN COMPARATIVE VISUALIZATIONS
# # -------------------------------------------------------------------------
# target_id = 3
# filter_size = 5

# # 1. Generate the globally filtered volume mask first
# globally_smoothed_mask = median_filter(muscles[target_id], size=9)

# total_slices = len(centerlines[target_id])

# # 2. Run the comparison on the target slice
# debug_global_mask_median_vs_angular(
#     centerline=centerlines[target_id],
#     raw_mask=muscles[target_id],
#     pre_filtered_mask=globally_smoothed_mask,
#     pixel_spacing=pixel_spacing,
#     slice_idx=total_slices // 2,
#     n_theta=64,
#     filter_size=filter_size
# )